In [ ]:
# =============================================================================
# Experiment 2: DDoS Attack Prediction using Time Series Forecasting (ARIMA / Prophet)
# Objective: Using historical network traffic data, apply time series forecasting methods (like ARIMA or Prophet) to predict potential DDoS attacks. [CO1]
# =============================================================================

# 1. Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error

# Styling for plots
plt.style.use('seaborn-v0_8' if 'seaborn-v0_8' in plt.style.available else 'default')
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# =============================================================================
# 2. Generate Synthetic Historical Network Traffic Data (Requests per Minute)
# Simulating normal baseline traffic with sudden spikes representing potential DDoS attacks.
# =============================================================================

np.random.seed(42)
time_index = pd.date_range(start='2026-11-01 00:00:00', periods=150, freq='min')

# Baseline normal traffic (sinusoidal pattern + noise)
baseline_traffic = 500 + 150 * np.sin(np.linspace(0, 10, 150)) + np.random.normal(0, 30, 150)

# Inject synthetic DDoS attack traffic spikes at specific timestamps
attack_spikes = np.zeros(150)
attack_spikes[45:52] = [800, 1400, 2200, 2500, 2100, 1300, 900]  # First attack wave
attack_spikes[110:116] = [1000, 1900, 2600, 2400, 1500, 950]  # Second attack wave

# Total network traffic
total_traffic = np.clip(baseline_traffic + attack_spikes, 100, None)

traffic_df = pd.DataFrame({'Traffic_Requests': total_traffic}, index=time_index)

# Display first few rows
print("--- First 5 Rows of Network Traffic Data ---")
print(traffic_df.head())

In [ ]:
# =============================================================================
# 3. Visualize Network Traffic and Attack Spikes
# =============================================================================

plt.figure(figsize=(14, 5))
plt.plot(traffic_df.index, traffic_df['Traffic_Requests'], color='crimson', linewidth=1.5, label='Network Traffic (Requests/min)')
plt.axhline(y=1200, color='orange', linestyle='--', label='DDoS Alert Threshold (1200 req/min)')
plt.title('Historical Network Traffic Monitoring & DDoS Detection', fontsize=14)
plt.xlabel('Timestamp', fontsize=12)
plt.ylabel('Requests per Minute', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 4. Train ARIMA Forecasting Model
# Splitting into train and test sets to forecast future traffic behavior and predict anomalies.
# =============================================================================

train_data = traffic_df['Traffic_Requests'].iloc[:120]
test_data = traffic_df['Traffic_Requests'].iloc[120:]

# Fit ARIMA model (Order: p=2, d=1, q=2)
model = ARIMA(train_data, order=(2, 1, 2))
fitted_model = model.fit()

# Forecast remaining periods
forecast_result = fitted_model.get_forecast(steps=len(test_data))
forecast_mean = forecast_result.predicted_mean
forecast_conf_int = forecast_result.conf_int()

print(fitted_model.summary().tables[0])

In [ ]:
# =============================================================================
# 5. Evaluate and Visualize DDoS Predictions
# Comparing forecasted traffic against actual traffic to flag potential attack anomalies.
# =============================================================================

plt.figure(figsize=(14, 6))
plt.plot(train_data.index, train_data, label='Training Traffic', color='blue')
plt.plot(test_data.index, test_data, label='Actual Traffic (Test)', color='crimson', linewidth=2)
plt.plot(test_data.index, forecast_mean, label='ARIMA Forecasted Traffic', color='green', linestyle='--', linewidth=2)

# Fill confidence interval
plt.fill_between(test_data.index, 
                 forecast_conf_int.iloc[:, 0], 
                 forecast_conf_int.iloc[:, 1], 
                 color='green', alpha=0.15, label='95% Confidence Interval')

plt.title('DDoS Attack Prediction using ARIMA Forecasting', fontsize=14)
plt.xlabel('Timestamp', fontsize=12)
plt.ylabel('Requests per Minute', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

# Compute RMSE
rmse = np.sqrt(mean_squared_error(test_data, forecast_mean))
print(f'Test RMSE: {rmse:.2f}')